# Biocad Tool

## Importing and Logging

In [8]:
from __future__ import annotations

import os
import warnings
import argparse
import logging
from typing import Dict, List, Tuple
from collections import defaultdict
from concurrent.futures import ProcessPoolExecutor

import numpy as np
from rdkit import Chem
from openff.toolkit import Molecule
from openff.toolkit.typing.engines.smirnoff import ForceField
from openff.interchange import Interchange
from tqdm import tqdm

# ENV
warnings.filterwarnings("ignore")

os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"

# LOGGING
def setup_logging(log_file: str) -> None:
    logging.basicConfig(
        level=logging.INFO,
        format="%(asctime)s [%(levelname)s] %(message)s",
        handlers=[
            logging.FileHandler(log_file),
            logging.StreamHandler()
        ]
    )

logger = logging.getLogger(__name__)

MolGroupKey = Tuple[str, str]
MolGroup = Dict[MolGroupKey, List[Chem.Mol]]


## Grouping

In [2]:
def group_molecules(sdf_path: str) -> MolGroup:
    supplier = Chem.SDMolSupplier(sdf_path, removeHs=False)
    groups: MolGroup = defaultdict(list)

    for idx, mol in enumerate(supplier):
        if mol is None:
            logger.warning(f"Skipping invalid molecule at index {idx}")
            continue

        try:
            Chem.SanitizeMol(mol)
        except Exception as e:
            logger.warning(f"Sanitize failed at {idx}: {e}")
            continue

        smiles = Chem.MolToSmiles(mol)
        name = mol.GetProp("_Name") if mol.HasProp("_Name") else f"mol_{idx}"

        groups[(name, smiles)].append(mol)

    logger.info(f"Grouped into {len(groups)} unique molecules")
    return groups

## Build interchange

In [3]:
def build_interchange(mol: Chem.Mol, mol_name: str) -> tuple[Molecule, Interchange]:
    force_field = ForceField("openff-2.1.0.offxml")

    off_mol = Molecule.from_rdkit(mol, allow_undefined_stereo=True)
    off_mol.name = mol_name

    topology = off_mol.to_topology()

    interchange = Interchange.from_smirnoff(
        force_field=force_field,
        topology=topology,
    )

    return off_mol, interchange

## Compute box

In [4]:
def compute_box(coords_nm: np.ndarray) -> np.ndarray:
    min_c = coords_nm.min(axis=0)
    max_c = coords_nm.max(axis=0)

    mol_size = np.max(max_c - min_c) + 2.0
    mol_size = max(mol_size, 2.0)

    return np.eye(3) * mol_size

## Worker

In [5]:
def extract_coords_nm(rdmol: Chem.Mol) -> np.ndarray:
    conf = rdmol.GetConformer()

    coords = np.array([
        conf.GetAtomPosition(i) for i in range(rdmol.GetNumAtoms())
    ])

    return coords / 10.0
def process_mol_group(args: Tuple[MolGroupKey, List[Chem.Mol], str]) -> None:
    (mol_name, smiles), mols, output_dir = args

    logger.info(f"Processing: {mol_name} ({len(mols)} conformers)")

    base_mol = mols[0]

    try:
        Chem.SanitizeMol(base_mol)
    except Exception as e:
        logger.error(f"Failed sanitize base mol {mol_name}: {e}")
        return

    off_mol, interchange = build_interchange(base_mol, mol_name)

    mol_dir = os.path.join(output_dir, mol_name)
    os.makedirs(mol_dir, exist_ok=True)

    # TOPOLOGY
    top_path = os.path.join(mol_dir, "topol.top")
    interchange.to_top(top_path)

    # CONFORMERS
    for i, rdmol in enumerate(mols):
        try:
            Chem.SanitizeMol(rdmol)
            coords_nm = extract_coords_nm(rdmol)

            if coords_nm.shape[0] != off_mol.n_atoms:
                logger.warning(
                    f"Atom mismatch in {mol_name}: {coords_nm.shape[0]} vs {off_mol.n_atoms}"
                )
                continue

            interchange.positions = coords_nm
            interchange.box = compute_box(coords_nm)

            gro_path = os.path.join(mol_dir, f"conf_{i}.gro")
            interchange.to_gro(gro_path)

        except Exception as e:
            logger.error(f"Failed conformer {i} in {mol_name}: {e}")

    logger.info(f"Done: {mol_name}")

## Paralleling 

In [6]:
# PARALLEL
def process_sdf_parallel(
    sdf_path: str,
    output_dir: str,
    n_workers: int,
) -> None:
    os.makedirs(output_dir, exist_ok=True)

    groups = group_molecules(sdf_path)

    tasks = [
        ((name, smiles), mols, output_dir)
        for (name, smiles), mols in groups.items()
    ]

    with ProcessPoolExecutor(max_workers=n_workers) as executor:
        list(tqdm(
            executor.map(process_mol_group, tasks),
            total=len(tasks),
            desc="Processing molecules"
        ))

## CLI + MAIN

In [7]:
def main() -> None:
    setup_logging('run.log')

    logger.info(f"Starting processing")

    process_sdf_parallel(
        sdf_path='../data/example.sdf',
        output_dir='ligands',
        n_workers=8,
    )

    logger.info("All done")


if __name__ == "__main__":
    main()


2026-03-27 16:22:53,309 [INFO] Starting processing
[16:22:53] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[16:22:53] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[16:22:53] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[16:22:53] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[16:22:53] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[16:22:53] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[16:22:53] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[16:22:53] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[16:22:53] Warning: molecule is tagged as 2D, but at 